# Granite 4.1 Benchmark — Results Overview

Loads cached results from Phase 1 (BFCL v3 tool calling) and Phase 2 (IFEval instruction following).

**No inference needed** — runs entirely from cached JSON files in `results/` and `results_ifeval/`.

In [1]:
import sys
sys.path.insert(0, '.')
import json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display
from config import MODELS, BFCL_CATEGORIES
from data.loader import load_ground_truth
from data.ifeval_loader import load_ifeval
from evaluation.metrics import evaluate_results, compute_category_metrics
from evaluation.ifeval_evaluator import compute_ifeval_metrics, evaluate_ifeval_response
pd.set_option('display.max_colwidth', None)
PALETTE = sns.color_palette('colorblind', len(MODELS))
COLORS = {m['label']: PALETTE[i] for i, m in enumerate(MODELS)}
LABELS = [m['label'] for m in MODELS]
print('All imports OK')
print(f'Models: {LABELS}')

All imports OK
Models: ['Granite 4.1 8B', 'Granite 4.0', 'Llama 3.1 8B', 'Qwen 2.5 7B', 'Mistral 7B']


## Phase 1 — Tool Calling (BFCL v3)

In [2]:
# Load Phase 1 cached results
all_raw_results = {}
for model in MODELS:
    tag, label = model['tag'], model['label']
    safe_tag = tag.replace(':', '_').replace('/', '_')
    all_raw_results[label] = {}
    for cat in BFCL_CATEGORIES:
        cat_dir = Path('results') / safe_tag
        files = sorted(cat_dir.glob(f'{cat}_*.json'))
        all_raw_results[label][cat] = [json.loads(f.read_text()) for f in files]
    total = sum(len(v) for v in all_raw_results[label].values())
    print(f'{label}: {total} results loaded')

# Compute Phase 1 metrics
all_metrics = {}
for model in MODELS:
    label = model['label']
    all_metrics[label] = {}
    for cat in BFCL_CATEGORIES:
        gt = load_ground_truth(cat)
        raw = all_raw_results[label][cat]
        evaled = evaluate_results(raw, gt, cat)
        all_metrics[label][cat] = compute_category_metrics(evaled)
print('Phase 1 metrics computed.')

Granite 4.1 8B: 1200 results loaded


Granite 4.0: 1200 results loaded
Llama 3.1 8B: 1200 results loaded


Qwen 2.5 7B: 1200 results loaded


Mistral 7B: 1200 results loaded
Phase 1 metrics computed.


In [3]:
# Phase 1 accuracy table
rows = []
for model in MODELS:
    label = model['label']
    m = all_metrics[label]
    total_c = sum(int(v.full_acc * v.total) for v in m.values())
    total_n = sum(v.total for v in m.values())
    row = {'Model': label}
    for cat in BFCL_CATEGORIES:
        row[cat.replace('_', ' ').title()] = f"{m[cat].full_acc:.1%}"
    row['Overall'] = f"{total_c/total_n:.1%}"
    rows.append(row)
df1 = pd.DataFrame(rows).set_index('Model')
print('=== Phase 1 — Tool Calling Accuracy (BFCL v3) ===')
display(df1)
print('\nIBM published Granite 4.1 8B BFCL v3 score = 68.27%')

=== Phase 1 — Tool Calling Accuracy (BFCL v3) ===


,Simple,Multiple,Parallel,Parallel Multiple,Overall
Model,,,,,
Granite 4.1 8B,50.2%,46.5%,30.0%,47.0%,42.2%
Granite 4.0,43.0%,39.5%,24.2%,34.5%,34.8%
Llama 3.1 8B,49.5%,51.5%,24.0%,44.0%,40.4%
Qwen 2.5 7B,53.0%,48.0%,28.0%,47.0%,42.8%
Mistral 7B,48.2%,41.0%,21.2%,29.5%,34.9%



IBM published Granite 4.1 8B BFCL v3 score = 68.27%


In [4]:
# Chart 1 — Overall accuracy
overall_accs = []
for model in MODELS:
    label = model['label']
    m = all_metrics[label]
    total_c = sum(int(v.full_acc * v.total) for v in m.values())
    total_n = sum(v.total for v in m.values())
    overall_accs.append(total_c / total_n * 100)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(LABELS, overall_accs, color=[COLORS[l] for l in LABELS], width=0.5, edgecolor='white')
ax.axhline(68.27, color='red', linestyle='--', alpha=0.7, label='IBM published (68.27%)')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
ax.set_ylabel('Overall AST Accuracy (%)')
ax.set_title('Phase 1 — Tool Calling Overall Accuracy (BFCL v3)')
ax.set_ylim(0, 80)
ax.tick_params(axis='x', rotation=15)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('charts/chart1_accuracy_comparison.png', dpi=150)
plt.show()
print('Saved chart1')

Saved chart1


In [5]:
# Chart 2 — Heatmap
data = np.array([[all_metrics[m['label']][c].full_acc * 100 for c in BFCL_CATEGORIES] for m in MODELS])
cat_labels = [c.replace('_', '\n').title() for c in BFCL_CATEGORIES]
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(data, annot=True, fmt='.1f', xticklabels=cat_labels, yticklabels=LABELS,
            cmap='YlGn', vmin=0, vmax=60, ax=ax, linewidths=0.5, cbar_kws={'label': 'Accuracy (%)'})
ax.set_title('Phase 1 — Accuracy Heatmap (Models x Categories)')
plt.tight_layout()
plt.savefig('charts/chart2_heatmap.png', dpi=150)
plt.show()
print('Saved chart2')

Saved chart2


In [6]:
# Chart 3 — Granite 4.1 vs 4.0 delta
deltas = [all_metrics['Granite 4.1 8B'][c].full_acc * 100 - all_metrics['Granite 4.0'][c].full_acc * 100 for c in BFCL_CATEGORIES]
cat_labels_short = [c.replace('_', ' ').title() for c in BFCL_CATEGORIES]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(cat_labels_short, deltas, color=['#2ecc71' if d > 0 else '#e74c3c' for d in deltas], width=0.5)
ax.bar_label(bars, fmt='+%.1f%%', padding=3, fontsize=11)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Accuracy Improvement (pp)')
ax.set_title('Phase 1 — Granite 4.1 8B vs Granite 4.0 (improvement per category)')
ax.set_ylim(-5, 20)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('charts/chart3_version_delta.png', dpi=150)
plt.show()
print('Saved chart3')

Saved chart3


In [7]:
# Chart 4 — Token efficiency
token_data = {}
for model in MODELS:
    label = model['label']
    correct_tokens = []
    for cat in BFCL_CATEGORIES:
        m = all_metrics[label][cat]
        if m.avg_tokens_correct > 0:
            correct_tokens.extend([m.avg_tokens_correct] * max(1, int(m.full_acc * m.total)))
    token_data[label] = np.mean(correct_tokens) if correct_tokens else 0

fig, ax = plt.subplots(figsize=(10, 5))
vals = [token_data[l] for l in LABELS]
bars = ax.bar(LABELS, vals, color=[COLORS[l] for l in LABELS], width=0.5, edgecolor='white')
ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=10)
ax.set_ylabel('Avg Completion Tokens (correct calls only)')
ax.set_title('Phase 1 — Token Efficiency')
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('charts/chart4_token_efficiency.png', dpi=150)
plt.show()
print('Saved chart4')

Saved chart4


In [8]:
# Chart 5 — Radar chart
top3 = ['Granite 4.1 8B', 'Llama 3.1 8B', 'Qwen 2.5 7B']
N = len(BFCL_CATEGORIES)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]
fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
for label in top3:
    values = [all_metrics[label][c].full_acc * 100 for c in BFCL_CATEGORIES] + [all_metrics[label][BFCL_CATEGORIES[0]].full_acc * 100]
    ax.plot(angles, values, 'o-', linewidth=2, label=label, color=COLORS[label])
    ax.fill(angles, values, alpha=0.1, color=COLORS[label])
ax.set_xticks(angles[:-1])
ax.set_xticklabels([c.replace('_', '\n').title() for c in BFCL_CATEGORIES], size=11)
ax.set_ylim(0, 60)
ax.set_title('Phase 1 — Capability Radar (Top 3 Models)', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('charts/chart5_radar.png', dpi=150)
plt.show()
print('Saved chart5')

Saved chart5


## Phase 2 — Instruction Following (IFEval)

In [9]:
# Load Phase 2 cached results
samples = load_ifeval()
sample_map = {s['id']: s for s in samples}
all_ifeval_results = {}
for model in MODELS:
    tag, label = model['tag'], model['label']
    safe_tag = tag.replace(':', '_').replace('/', '_')
    model_dir = Path('results_ifeval') / safe_tag
    results = [json.loads(f.read_text()) for f in sorted(model_dir.glob('ifeval_*.json'))]
    all_ifeval_results[label] = results
    print(f'{label}: {len(results)} results loaded')
all_ifeval_metrics = {m['label']: compute_ifeval_metrics(all_ifeval_results[m['label']], samples) for m in MODELS}
print('Phase 2 metrics computed.')

Granite 4.1 8B: 541 results loaded


Granite 4.0: 541 results loaded


Llama 3.1 8B: 541 results loaded


Qwen 2.5 7B: 541 results loaded


Mistral 7B: 541 results loaded


Phase 2 metrics computed.


In [10]:
# Phase 2 summary table
rows = []
for model in MODELS:
    label = model['label']
    m = all_ifeval_metrics[label]
    rows.append({'Model': label, 'Prompt Accuracy': f"{m.prompt_strict_acc:.1%}",
                 'Instruction Accuracy': f"{m.instruction_strict_acc:.1%}",
                 'Avg Tokens': round(m.avg_tokens), 'Median Latency (ms)': round(m.median_latency_ms)})
df2 = pd.DataFrame(rows).set_index('Model')
print('=== Phase 2 — Instruction Following (IFEval) ===')
display(df2)
print('\nIBM published Granite 4.1 8B IFEval score = 87.06%')

=== Phase 2 — Instruction Following (IFEval) ===


,Prompt Accuracy,Instruction Accuracy,Avg Tokens,Median Latency (ms)
Model,,,,
Granite 4.1 8B,79.7%,85.3%,299,9531
Granite 4.0,77.1%,83.6%,277,4009
Llama 3.1 8B,70.6%,78.2%,299,10611
Qwen 2.5 7B,70.4%,78.7%,271,8022
Mistral 7B,46.8%,56.8%,345,11399



IBM published Granite 4.1 8B IFEval score = 87.06%


In [11]:
# Chart — Phase 2 accuracy
prompt_accs = [all_ifeval_metrics[m['label']].prompt_strict_acc * 100 for m in MODELS]
instr_accs  = [all_ifeval_metrics[m['label']].instruction_strict_acc * 100 for m in MODELS]
x = np.arange(len(LABELS))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - width/2, prompt_accs, width, label='Prompt Accuracy', color=[COLORS[l] for l in LABELS], alpha=0.95)
b2 = ax.bar(x + width/2, instr_accs,  width, label='Instruction Accuracy', color=[COLORS[l] for l in LABELS], alpha=0.55)
ax.bar_label(b1, fmt='%.1f%%', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.1f%%', padding=3, fontsize=9)
ax.axhline(87.06, color='red', linestyle='--', alpha=0.7, label='IBM published (87.06%)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Phase 2 — Instruction Following Accuracy (IFEval)')
ax.set_xticks(x)
ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results_ifeval/ifeval_accuracy.png', dpi=150)
plt.show()
print('Saved ifeval_accuracy.png')

Saved ifeval_accuracy.png


In [12]:
# Phase 2 — category breakdown
from collections import defaultdict
categories = ['language','detectable_content','detectable_format','startend',
              'punctuation','change_case','keywords','length_constraints','combination']
cat_rows = []
for cat in categories:
    row = {'Category': cat}
    for model in MODELS:
        label = model['label']
        vals = []
        for r in all_ifeval_results[label]:
            if r.get('error'): continue
            s = sample_map.get(r['id'])
            if not s: continue
            ev = evaluate_ifeval_response(r['response'], s['instruction_id_list'], s['kwargs'])
            for instr_id, passed in zip(s['instruction_id_list'], ev.instruction_results):
                if instr_id.split(':')[0] == cat:
                    vals.append(passed)
        row[label] = f"{sum(vals)/len(vals):.1%}" if vals else 'N/A'
    cat_rows.append(row)
df_cat = pd.DataFrame(cat_rows).set_index('Category')
print('=== Phase 2 — Accuracy by Instruction Category ===')
display(df_cat)

=== Phase 2 — Accuracy by Instruction Category ===


,Granite 4.1 8B,Granite 4.0,Llama 3.1 8B,Qwen 2.5 7B,Mistral 7B
Category,,,,,
language,100.0%,93.5%,90.3%,100.0%,74.2%
detectable_content,96.2%,94.3%,82.4%,90.6%,83.0%
detectable_format,96.1%,94.9%,82.1%,87.2%,75.8%
startend,95.5%,97.0%,90.9%,88.1%,69.7%
punctuation,90.8%,92.4%,90.9%,93.9%,9.1%
change_case,85.1%,88.8%,77.3%,71.9%,54.5%
keywords,83.9%,76.1%,77.3%,75.2%,68.7%
length_constraints,73.0%,71.3%,68.8%,62.9%,45.1%
combination,70.3%,60.0%,66.2%,70.8%,18.5%


In [13]:
# Chart — Category heatmap
cat_data = np.array([[float(df_cat.loc[cat, m['label']].strip('%')) for m in MODELS] for cat in categories])
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(cat_data, annot=True, fmt='.1f', xticklabels=LABELS, yticklabels=categories,
            cmap='YlGn', vmin=0, vmax=100, ax=ax, linewidths=0.5, cbar_kws={'label': 'Accuracy (%)'})
ax.set_title('Phase 2 — Instruction Accuracy by Category and Model')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('results_ifeval/ifeval_category_heatmap.png', dpi=150)
plt.show()
print('Saved ifeval_category_heatmap.png')

Saved ifeval_category_heatmap.png


## Cross-Phase Comparison

In [14]:
# Cross-phase summary table
rows = []
for model in MODELS:
    label = model['label']
    m1 = all_metrics[label]
    total_c = sum(int(v.full_acc * v.total) for v in m1.values())
    total_n = sum(v.total for v in m1.values())
    m2 = all_ifeval_metrics[label]
    rows.append({'Model': label, 'Phase 1 Tool Calling': f"{total_c/total_n:.1%}",
                 'Phase 2 Prompt Acc': f"{m2.prompt_strict_acc:.1%}",
                 'Phase 2 Instr Acc': f"{m2.instruction_strict_acc:.1%}"})
df_cross = pd.DataFrame(rows).set_index('Model')
print('=== Cross-Phase Comparison ===')
display(df_cross)

=== Cross-Phase Comparison ===


,Phase 1 Tool Calling,Phase 2 Prompt Acc,Phase 2 Instr Acc
Model,,,
Granite 4.1 8B,42.2%,79.7%,85.3%
Granite 4.0,34.8%,77.1%,83.6%
Llama 3.1 8B,40.4%,70.6%,78.2%
Qwen 2.5 7B,42.8%,70.4%,78.7%
Mistral 7B,34.9%,46.8%,56.8%


In [15]:
# Chart — Cross-phase grouped bars
phase1_accs, phase2_prompt, phase2_instr = [], [], []
for model in MODELS:
    label = model['label']
    m1 = all_metrics[label]
    total_c = sum(int(v.full_acc * v.total) for v in m1.values())
    total_n = sum(v.total for v in m1.values())
    phase1_accs.append(total_c / total_n * 100)
    phase2_prompt.append(all_ifeval_metrics[label].prompt_strict_acc * 100)
    phase2_instr.append(all_ifeval_metrics[label].instruction_strict_acc * 100)

x = np.arange(len(LABELS))
width = 0.25
fig, ax = plt.subplots(figsize=(13, 6))
b1 = ax.bar(x - width, phase1_accs,   width, label='Phase 1 Tool Calling',  color=[COLORS[l] for l in LABELS], alpha=0.95)
b2 = ax.bar(x,          phase2_prompt, width, label='Phase 2 Prompt Acc',    color=[COLORS[l] for l in LABELS], alpha=0.65)
b3 = ax.bar(x + width,  phase2_instr,  width, label='Phase 2 Instr Acc',     color=[COLORS[l] for l in LABELS], alpha=0.35)
ax.bar_label(b1, fmt='%.1f%%', padding=2, fontsize=8)
ax.bar_label(b2, fmt='%.1f%%', padding=2, fontsize=8)
ax.bar_label(b3, fmt='%.1f%%', padding=2, fontsize=8)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Phase 1 vs Phase 2 — All Models Comparison')
ax.set_xticks(x)
ax.set_xticklabels(LABELS, rotation=15)
ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results_ifeval/cross_phase_comparison.png', dpi=150)
plt.show()
print('Saved cross_phase_comparison.png')

Saved cross_phase_comparison.png


In [16]:
# Rank shift table
p1_sorted = sorted(LABELS, key=lambda l: -phase1_accs[LABELS.index(l)])
p2_sorted = sorted(LABELS, key=lambda l: -phase2_prompt[LABELS.index(l)])
rows = []
for label in LABELS:
    p1r = p1_sorted.index(label) + 1
    p2r = p2_sorted.index(label) + 1
    shift = p1r - p2r
    arrow = ('up ' + str(abs(shift))) if shift > 0 else (('down ' + str(abs(shift))) if shift < 0 else 'same')
    rows.append({'Model': label, 'Phase 1 Rank': p1r, 'Phase 2 Rank': p2r, 'Shift': arrow})
df_rank = pd.DataFrame(rows).set_index('Model')
print('=== Rank Shifts Between Phase 1 and Phase 2 ===')
display(df_rank)

=== Rank Shifts Between Phase 1 and Phase 2 ===


,Phase 1 Rank,Phase 2 Rank,Shift
Model,,,
Granite 4.1 8B,2,1,up 1
Granite 4.0,5,2,up 3
Llama 3.1 8B,3,3,same
Qwen 2.5 7B,1,4,down 3
Mistral 7B,4,5,down 1
